# Hotel Customer Returns Prediction Model

This notebook demonstrates a complete ML pipeline for predicting whether hotel customers will return:
1. Generate synthetic hotel customer data
2. Perform data preparation and feature engineering
3. Train a classification model
4. Deploy the model to IBM Cloud Pak for Data

## 1. Environment Setup

In [23]:
!pip install ibm-watsonx-ai --quiet

In [24]:
import os
import json
import numpy as np
import pandas as pd
import joblib  # Use joblib instead of pickle for sklearn models
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.pipeline import Pipeline

# IBM Watson Machine Learning
# from ibm_watson_machine_learning import APIClient

from ibm_watsonx_ai import APIClient, Credentials

print("Libraries imported successfully")

Libraries imported successfully


In [25]:
# Pipeline credentials
os.environ["cloud_api_key"]   = "SZvJCsKOZhPnWhg6zjXs69VHxsDoMidhZ76aFDx9VWpg"
os.environ["PROJECT_ID"]      = "f7e292f1-571b-49c9-b9e4-d8df073ec50f"
os.environ["model_name"]      = "HotelCustomerReturnsModel"
os.environ["deployment_name"] = "HotelReturnsDeployment"
os.environ["feature_pickle"]  = "hotel_feature_pipeline.pkl"
os.environ["catalog_name"]    = "MLOps-ns"
os.environ["space_id"]        = "78a08b44-12d1-46fe-8da9-7450ce94acd2"

# COS credentials
os.environ["project_cos_credentials"] = json.dumps({
    "AUTH_ENDPOINT": "https://iam.cloud.ibm.com/oidc/token",
    "ENDPOINT_URL":  "https://s3.private.us.cloud-object-storage.appdomain.cloud",
    "API_KEY":       "lQbSPOZ2TczYOmbXxruMVp6pXjUg8mmYrpEypg-KJ3yB",
    "BUCKET":        "ww-cpd-mlops-project-cos"
})

os.environ["mlops_cos_credentials"] = json.dumps({
    "ENDPOINT_URL": "https://s3.private.us.cloud-object-storage.appdomain.cloud",
    "API_KEY":      "lQbSPOZ2TczYOmbXxruMVp6pXjUg8mmYrpEypg-KJ3yB",
    "CRN":          "crn:v1:bluemix:public:cloud-object-storage:global:a/7cd77b117ea34e7e8bf3ed8da0d3080f:c0f7b49b-5a9a-4c5c-b35d-0abd9bdbed59::",
    "BUCKET":       "ww-cpd-mlops-mlops-cos"
})

print("Environment variables set successfully")

Environment variables set successfully


## 2. Generate Sample Hotel Customer Data

In [26]:
def generate_hotel_customer_data(n_samples=5000, random_state=42):
    """
    Generate synthetic hotel customer data with features that influence return likelihood
    """
    np.random.seed(random_state)
    
    # Customer demographics
    customer_ids = [f"CUST_{i:05d}" for i in range(n_samples)]
    ages = np.random.normal(45, 15, n_samples).clip(18, 85).astype(int)
    customer_types = np.random.choice(['Business', 'Leisure', 'Group'], n_samples, p=[0.3, 0.5, 0.2])
    
    # Booking characteristics
    booking_lead_time = np.random.exponential(30, n_samples).clip(0, 365).astype(int)
    length_of_stay = np.random.poisson(3, n_samples).clip(1, 14)
    num_previous_stays = np.random.poisson(2, n_samples).clip(0, 20)
    
    # Service usage
    room_type = np.random.choice(['Standard', 'Deluxe', 'Suite'], n_samples, p=[0.6, 0.3, 0.1])
    special_requests = np.random.poisson(1, n_samples).clip(0, 5)
    used_spa = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
    used_restaurant = np.random.choice([0, 1], n_samples, p=[0.4, 0.6])
    
    # Financial
    total_spend = np.random.gamma(2, 150, n_samples).clip(100, 5000)
    discount_used = np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
    
    # Experience indicators
    complaint_filed = np.random.choice([0, 1], n_samples, p=[0.85, 0.15])
    review_score = np.random.normal(4.0, 0.8, n_samples).clip(1, 5)
    response_time_hours = np.random.exponential(2, n_samples).clip(0.1, 24)
    
    # Season
    season = np.random.choice(['Spring', 'Summer', 'Fall', 'Winter'], n_samples)
    
    # Create target variable (will_return) based on features
    # Higher probability of return if:
    # - High review score
    # - Previous stays
    # - No complaints
    # - Used services
    # - Business travelers
    
    return_probability = (
        0.2 +  # Base probability
        (review_score - 1) * 0.15 +  # Review impact
        (num_previous_stays > 0) * 0.2 +  # Loyalty impact
        (1 - complaint_filed) * 0.15 +  # No complaint bonus
        (used_spa + used_restaurant) * 0.1 +  # Service usage
        (customer_types == 'Business') * 0.1 +  # Business traveler bonus
        (room_type == 'Suite') * 0.05 -  # Premium room bonus
        (response_time_hours > 5) * 0.1  # Slow response penalty
    )
    
    return_probability = np.clip(return_probability, 0, 1)
    will_return = (np.random.random(n_samples) < return_probability).astype(int)
    
    # Create DataFrame
    data = pd.DataFrame({
        'customer_id': customer_ids,
        'age': ages,
        'customer_type': customer_types,
        'booking_lead_time': booking_lead_time,
        'length_of_stay': length_of_stay,
        'num_previous_stays': num_previous_stays,
        'room_type': room_type,
        'special_requests': special_requests,
        'used_spa': used_spa,
        'used_restaurant': used_restaurant,
        'total_spend': total_spend,
        'discount_used': discount_used,
        'complaint_filed': complaint_filed,
        'review_score': review_score,
        'response_time_hours': response_time_hours,
        'season': season,
        'will_return': will_return
    })
    
    return data

# Generate data
df = generate_hotel_customer_data(n_samples=5000)
print(f"Generated {len(df)} customer records")
print(f"\nReturn rate: {df['will_return'].mean():.2%}")
df.head(10)

Generated 5000 customer records

Return rate: 95.96%


,customer_id,age,customer_type,booking_lead_time,length_of_stay,num_previous_stays,room_type,special_requests,used_spa,used_restaurant,total_spend,discount_used,complaint_filed,review_score,response_time_hours,season,will_return
0,CUST_00000,52,Business,11,3,1,Standard,2,0,0,267.726512,0,0,2.968732,0.231975,Winter,1
1,CUST_00001,42,Business,5,1,1,Suite,2,1,1,100.000000,1,0,4.480729,1.980397,Summer,1
2,CUST_00002,54,Leisure,58,5,1,Standard,1,1,0,358.980475,1,0,2.652237,3.063618,Fall,1
3,CUST_00003,67,Business,15,1,3,Standard,1,1,1,289.213119,1,0,4.757705,0.169000,Winter,1
4,CUST_00004,41,Business,15,2,3,Standard,0,1,1,100.000000,0,0,3.825328,0.500285,Winter,1
5,CUST_00005,41,Leisure,5,1,3,Suite,0,0,1,423.164287,0,0,4.647070,0.774528,Winter,1
6,CUST_00006,68,Leisure,14,3,3,Standard,0,1,0,396.203501,0,1,4.360550,5.521884,Summer,1
7,CUST_00007,56,Group,61,4,1,Standard,1,1,0,119.133572,1,0,5.000000,1.822819,Fall,1
8,CUST_00008,37,Leisure,13,2,3,Deluxe,1,0,0,375.075477,0,0,4.081204,0.226947,Summer,1
9,CUST_00009,53,Business,90,3,3,Standard,0,1,0,567.443977,0,0,5.000000,0.923256,Summer,1


In [27]:
# Data overview
print("Dataset Info:")
print(df.info())
print("\nBasic Statistics:")
print(df.describe())
print("\nTarget Distribution:")
print(df['will_return'].value_counts())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          5000 non-null   object 
 1   age                  5000 non-null   int64  
 2   customer_type        5000 non-null   object 
 3   booking_lead_time    5000 non-null   int64  
 4   length_of_stay       5000 non-null   int64  
 5   num_previous_stays   5000 non-null   int64  
 6   room_type            5000 non-null   object 
 7   special_requests     5000 non-null   int64  
 8   used_spa             5000 non-null   int64  
 9   used_restaurant      5000 non-null   int64  
 10  total_spend          5000 non-null   float64
 11  discount_used        5000 non-null   int64  
 12  complaint_filed      5000 non-null   int64  
 13  review_score         5000 non-null   float64
 14  response_time_hours  5000 non-null   float64
 15  season               500

## 3. Data Preparation and Feature Engineering

In [28]:
# Feature engineering
def engineer_features(df):
    """
    Create additional features from existing data
    """
    df = df.copy()
    
    # Spending per night
    df['spend_per_night'] = df['total_spend'] / df['length_of_stay']
    
    # Is repeat customer
    df['is_repeat_customer'] = (df['num_previous_stays'] > 0).astype(int)
    
    # Service engagement score
    df['service_engagement'] = df['used_spa'] + df['used_restaurant'] + (df['special_requests'] > 0).astype(int)
    
    # Customer value segment
    df['high_value'] = (df['total_spend'] > df['total_spend'].quantile(0.75)).astype(int)
    
    # Booking behavior
    df['early_booker'] = (df['booking_lead_time'] > 30).astype(int)
    
    # Experience quality
    df['positive_experience'] = ((df['review_score'] >= 4) & (df['complaint_filed'] == 0)).astype(int)
    
    return df

df_engineered = engineer_features(df)
print("Feature engineering completed")
print(f"Total features: {len(df_engineered.columns)}")
df_engineered.head()

Feature engineering completed
Total features: 23


,customer_id,age,customer_type,booking_lead_time,length_of_stay,num_previous_stays,room_type,special_requests,used_spa,used_restaurant,...,review_score,response_time_hours,season,will_return,spend_per_night,is_repeat_customer,service_engagement,high_value,early_booker,positive_experience
0,CUST_00000,52,Business,11,3,1,Standard,2,0,0,...,2.968732,0.231975,Winter,1,89.242171,1,1,0,0,0
1,CUST_00001,42,Business,5,1,1,Suite,2,1,1,...,4.480729,1.980397,Summer,1,100.000000,1,3,0,0,1
2,CUST_00002,54,Leisure,58,5,1,Standard,1,1,0,...,2.652237,3.063618,Fall,1,71.796095,1,2,0,1,0
3,CUST_00003,67,Business,15,1,3,Standard,1,1,1,...,4.757705,0.169000,Winter,1,289.213119,1,3,0,0,1
4,CUST_00004,41,Business,15,2,3,Standard,0,1,1,...,3.825328,0.500285,Winter,1,50.000000,1,2,0,0,0


In [29]:
# Prepare features for modeling
def prepare_features(df):
    """
    Encode categorical variables and prepare feature matrix
    """
    df = df.copy()
    
    # Encode categorical variables
    le_customer_type = LabelEncoder()
    le_room_type = LabelEncoder()
    le_season = LabelEncoder()
    
    df['customer_type_encoded'] = le_customer_type.fit_transform(df['customer_type'])
    df['room_type_encoded'] = le_room_type.fit_transform(df['room_type'])
    df['season_encoded'] = le_season.fit_transform(df['season'])
    
    # Store encoders for later use
    encoders = {
        'customer_type': le_customer_type,
        'room_type': le_room_type,
        'season': le_season
    }
    
    return df, encoders

df_prepared, encoders = prepare_features(df_engineered)
print("Feature preparation completed")

Feature preparation completed


In [30]:
# Select features for modeling
feature_columns = [
    'age', 'customer_type_encoded', 'booking_lead_time', 'length_of_stay',
    'num_previous_stays', 'room_type_encoded', 'special_requests',
    'used_spa', 'used_restaurant', 'total_spend', 'discount_used',
    'complaint_filed', 'review_score', 'response_time_hours', 'season_encoded',
    'spend_per_night', 'is_repeat_customer', 'service_engagement',
    'high_value', 'early_booker', 'positive_experience'
]

X = df_prepared[feature_columns]
y = df_prepared['will_return']

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeatures used: {len(feature_columns)}")

Feature matrix shape: (5000, 21)
Target shape: (5000,)

Features used: 21


## 4. Train-Test Split

In [31]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining set return rate: {y_train.mean():.2%}")
print(f"Test set return rate: {y_test.mean():.2%}")

Training set size: 4000
Test set size: 1000

Training set return rate: 95.95%
Test set return rate: 96.00%


## 5. Model Training

In [32]:
# Create and train model pipeline
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training model...")
model_pipeline.fit(X_train, y_train)
print("Model training completed!")

Training model...
Model training completed!


## 6. Model Evaluation

In [33]:
# Make predictions
y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)
y_test_proba = model_pipeline.predict_proba(X_test)[:, 1]

# Calculate metrics
def evaluate_model(y_true, y_pred, y_proba=None, dataset_name=""):
    print(f"\n{'='*50}")
    print(f"{dataset_name} Performance Metrics")
    print(f"{'='*50}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1 Score:  {f1_score(y_true, y_pred):.4f}")
    if y_proba is not None:
        print(f"ROC AUC:   {roc_auc_score(y_true, y_proba):.4f}")
    
    print(f"\nConfusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(f"  Predicted No | Predicted Yes")
    print(f"Actual No:  {cm[0][0]:4d}     |  {cm[0][1]:4d}")
    print(f"Actual Yes: {cm[1][0]:4d}     |  {cm[1][1]:4d}")

evaluate_model(y_train, y_train_pred, dataset_name="Training Set")
evaluate_model(y_test, y_test_pred, y_test_proba, dataset_name="Test Set")


Training Set Performance Metrics
Accuracy:  0.9615
Precision: 0.9614
Recall:    1.0000
F1 Score:  0.9803

Confusion Matrix:
  Predicted No | Predicted Yes
Actual No:     8     |   154
Actual Yes:    0     |  3838

Test Set Performance Metrics
Accuracy:  0.9600
Precision: 0.9600
Recall:    1.0000
F1 Score:  0.9796
ROC AUC:   0.9026

Confusion Matrix:
  Predicted No | Predicted Yes
Actual No:     0     |    40
Actual Yes:    0     |   960


In [34]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model_pipeline.named_steps['classifier'].feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10).to_string(index=False))


Top 10 Most Important Features:
              feature  importance
         review_score    0.210702
   num_previous_stays    0.088516
          total_spend    0.085776
  response_time_hours    0.084558
      spend_per_night    0.071854
   is_repeat_customer    0.069390
                  age    0.064934
    booking_lead_time    0.057133
  positive_experience    0.037694
customer_type_encoded    0.036727


## 7. Save Model and Artifacts

In [35]:
# Save the model pipeline using joblib (required by Watson ML)
model_filename = 'hotel_model_pipeline.joblib'

# Save just the model pipeline (Watson ML expects the model object)
joblib.dump(model_pipeline, model_filename)

print(f"Model pipeline saved to {model_filename}")
print(f"File size: {os.path.getsize(model_filename) / 1024:.2f} KB")

# Also save complete artifacts separately for reference
pipeline_artifacts = {
    'model_pipeline': model_pipeline,
    'encoders': encoders,
    'feature_columns': feature_columns,
    'training_date': datetime.now().isoformat(),
    'model_metrics': {
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_precision': precision_score(y_test, y_test_pred),
        'test_recall': recall_score(y_test, y_test_pred),
        'test_f1': f1_score(y_test, y_test_pred),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba)
    }
}

# Save artifacts for reference
artifacts_filename = 'hotel_artifacts.joblib'
joblib.dump(pipeline_artifacts, artifacts_filename)
print(f"Artifacts saved to {artifacts_filename}")

Model pipeline saved to hotel_model_pipeline.joblib
File size: 1276.02 KB
Artifacts saved to hotel_artifacts.joblib


## 8. Deploy Model to IBM Watson Machine Learning

In [36]:
# Initialize Watson Machine Learning client
wml_credentials = {
    "url": "https://us-south.ml.cloud.ibm.com",
    "apikey": os.environ.get('cloud_api_key')
}

client = APIClient(wml_credentials)
print("Watson Machine Learning client initialized")
print(f"Client version: {client.version}")

Watson Machine Learning client initialized
Client version: 1.5.5


In [37]:
# Set default space
space_id = os.environ.get('space_id')
client.set.default_space(space_id)
print(f"Default space set to: {space_id}")

Default space set to: 78a08b44-12d1-46fe-8da9-7450ce94acd2


In [38]:
# Define model metadata with compatible runtime
model_name = os.environ.get('model_name', 'HotelCustomerReturnsModel')

# Use runtime-24.1-py3.11 which is compatible with scikit-learn_1.3
# This runtime is deprecated but still functional and compatible
runtime_id = '45f12dfe-aa78-5b8d-9f38-0ee223c47309'  # runtime-24.1-py3.11

model_metadata = {
    client.repository.ModelMetaNames.NAME: model_name,
    client.repository.ModelMetaNames.TYPE: 'scikit-learn_1.3',
    client.repository.ModelMetaNames.SOFTWARE_SPEC_UID: runtime_id
}

print(f"Model metadata prepared for: {model_name}")
print(f"Using runtime: runtime-24.1-py3.11 (compatible with scikit-learn_1.3)")

Model metadata prepared for: HotelCustomerReturnsModel
Using runtime: runtime-24.1-py3.11 (compatible with scikit-learn_1.3)


In [39]:
# Store model in Watson Machine Learning
print("Storing model in Watson Machine Learning...")

# Watson ML SDK expects joblib-serialized sklearn models
# Pass the model pipeline directly (already joblib-compatible)
published_model = client.repository.store_model(
    model=model_pipeline,
    meta_props=model_metadata
)

model_uid = client.repository.get_model_id(published_model)
print(f"Model stored successfully!")
print(f"Model UID: {model_uid}")

Storing model in Watson Machine Learning...
Model stored successfully!
Model UID: 2586c71e-c98d-423d-ad19-bf2b41b561b1


In [40]:
# Create deployment
deployment_name = os.environ.get('deployment_name', 'HotelReturnsDeployment')

deployment_metadata = {
    client.deployments.ConfigurationMetaNames.NAME: deployment_name,
    client.deployments.ConfigurationMetaNames.ONLINE: {}
}

print(f"Creating deployment: {deployment_name}...")
deployment = client.deployments.create(model_uid, meta_props=deployment_metadata)
deployment_uid = client.deployments.get_id(deployment)

print(f"Deployment created successfully!")
print(f"Deployment UID: {deployment_uid}")

Creating deployment: HotelReturnsDeployment...


######################################################################################

Synchronous deployment creation for id: '2586c71e-c98d-423d-ad19-bf2b41b561b1' started

######################################################################################


initializing
Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html
.............
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='d79d1939-890c-41eb-a132-43ecc3e1657d'
-----------------------------------------------------------------------------------------------


Deployment created successfully!

In [41]:
# Get deployment details
deployment_details = client.deployments.get_details(deployment_uid)
scoring_endpoint = client.deployments.get_scoring_href(deployment_details)

print(f"\n{'='*70}")
print("DEPLOYMENT SUMMARY")
print(f"{'='*70}")
print(f"Model Name:        {model_name}")
print(f"Deployment Name:   {deployment_name}")
print(f"Model UID:         {model_uid}")
print(f"Deployment UID:    {deployment_uid}")
print(f"Scoring Endpoint:  {scoring_endpoint}")
print(f"Status:            {deployment_details['entity']['status']['state']}")
print(f"{'='*70}")

Note: Scikit-learn 1.3/xgboost 2.0 framework with runtime-24.1-py3.11 for Watson Machine Learning is deprecated and will be removed in future. Use Scikit-learn 1.6/XGBoost 2.1 with runtime-25.1-py3.12 instead. For details, see https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/pm_service_supported_frameworks.html

DEPLOYMENT SUMMARY
Model Name:        HotelCustomerReturnsModel
Deployment Name:   HotelReturnsDeployment
Model UID:         2586c71e-c98d-423d-ad19-bf2b41b561b1
Deployment UID:    d79d1939-890c-41eb-a132-43ecc3e1657d
Scoring Endpoint:  https://us-south.ml.cloud.ibm.com/ml/v4/deployments/d79d1939-890c-41eb-a132-43ecc3e1657d/predictions
Status:            ready


## 9. Test Deployment with Sample Predictions

In [42]:
# Prepare test payload
test_samples = X_test.head(5)
test_payload = {
    client.deployments.ScoringMetaNames.INPUT_DATA: [{
        'fields': feature_columns,
        'values': test_samples.values.tolist()
    }]
}

print("Test payload prepared with 5 samples")

Test payload prepared with 5 samples


In [43]:
# Score the model
print("Scoring model with test samples...")
predictions = client.deployments.score(deployment_uid, test_payload)

print("\nPrediction Results:")
print(json.dumps(predictions, indent=2))

Scoring model with test samples...

Prediction Results:
{
  "predictions": [
    {
      "fields": [
        "prediction",
        "probability"
      ],
      "values": [
        [
          1,
          [
            0.0062085187539733,
            0.9937914812460267
          ]
        ],
        [
          1,
          [
            0.08892003455656869,
            0.9110799654434313
          ]
        ],
        [
          1,
          [
            0.1146206322139718,
            0.885379367786028
          ]
        ],
        [
          1,
          [
            0.08640159930905676,
            0.913598400690943
          ]
        ],
        [
          1,
          [
            0.0069201307833462004,
            0.993079869216654
          ]
        ]
      ]
    }
  ]
}


In [44]:
# Display predictions in a readable format
if 'predictions' in predictions:
    pred_values = predictions['predictions'][0]['values']
    
    results_df = pd.DataFrame({
        'Sample': range(1, len(pred_values) + 1),
        'Predicted_Return': [p[0] for p in pred_values],
        'Probability_No_Return': [p[1][0] for p in pred_values],
        'Probability_Return': [p[1][1] for p in pred_values]
    })
    
    print("\nFormatted Predictions:")
    print(results_df.to_string(index=False))
else:
    print("Predictions format may differ. Raw output shown above.")


Formatted Predictions:
 Sample  Predicted_Return  Probability_No_Return  Probability_Return
      1                 1               0.006209            0.993791
      2                 1               0.088920            0.911080
      3                 1               0.114621            0.885379
      4                 1               0.086402            0.913598
      5                 1               0.006920            0.993080


## 10. Summary and Next Steps

In [45]:
print("\n" + "="*70)
print("HOTEL CUSTOMER RETURNS PREDICTION - PROJECT SUMMARY")
print("="*70)
print(f"\n✓ Generated {len(df)} synthetic hotel customer records")
print(f"✓ Engineered {len(feature_columns)} features for prediction")
print(f"✓ Trained Random Forest model with {len(X_train)} samples")
print(f"✓ Achieved test accuracy: {accuracy_score(y_test, y_test_pred):.2%}")
print(f"✓ Achieved test ROC AUC: {roc_auc_score(y_test, y_test_proba):.4f}")
print(f"✓ Model deployed to IBM Watson Machine Learning")
print(f"✓ Deployment is ready for scoring")

print("\nKey Features Influencing Customer Returns:")
for idx, row in feature_importance.head(5).iterrows():
    print(f"  {idx+1}. {row['feature']}: {row['importance']:.4f}")

print("\nNext Steps:")
print("  1. Monitor model performance in production")
print("  2. Collect feedback and retrain with real data")
print("  3. Implement A/B testing for model improvements")
print("  4. Create dashboards for business insights")
print("  5. Set up automated retraining pipeline")
print("="*70)


HOTEL CUSTOMER RETURNS PREDICTION - PROJECT SUMMARY

✓ Generated 5000 synthetic hotel customer records
✓ Engineered 21 features for prediction
✓ Trained Random Forest model with 4000 samples
✓ Achieved test accuracy: 96.00%
✓ Achieved test ROC AUC: 0.9026
✓ Model deployed to IBM Watson Machine Learning
✓ Deployment is ready for scoring

Key Features Influencing Customer Returns:
  13. review_score: 0.2107
  5. num_previous_stays: 0.0885
  10. total_spend: 0.0858
  14. response_time_hours: 0.0846
  16. spend_per_night: 0.0719

Next Steps:
  1. Monitor model performance in production
  2. Collect feedback and retrain with real data
  3. Implement A/B testing for model improvements
  4. Create dashboards for business insights
  5. Set up automated retraining pipeline
